<a href="https://colab.research.google.com/github/kavyavalleti/NeuroNexus/blob/main/titanic_survival_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving titanic_model_v2.pkl to titanic_model_v2.pkl


In [ ]:
import os
print(os.listdir())


['.config', 'titanic_model_v2.pkl', 'titanic_model (1).pkl', 'titanic_model.pkl', 'sample_data']


In [ ]:
import joblib
model = joblib.load("titanic_model_v2.pkl")


/usr/local/lib/python3.11/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.1.3 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.1.3 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.1.3 when using version 1.6.1. This might lead to breaking code or invalid

ValueError: node array from the pickle has an incompatible dtype:
- expected: {'names': ['left_child', 'right_child', 'feature', 'threshold', 'impurity', 'n_node_samples', 'weighted_n_node_samples', 'missing_go_to_left'], 'formats': ['<i8', '<i8', '<i8', '<f8', '<f8', '<i8', '<f8', 'u1'], 'offsets': [0, 8, 16, 24, 32, 40, 48, 56], 'itemsize': 64}
- got     : [('left_child', '<i8'), ('right_child', '<i8'), ('feature', '<i8'), ('threshold', '<f8'), ('impurity', '<f8'), ('n_node_samples', '<i8'), ('weighted_n_node_samples', '<f8')]

In [ ]:
!pip install gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.9/322.9 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 4.3 MB/s eta 0:00:00


In [ ]:

!pip install gradio --quiet


import pandas as pd
import gradio as gr
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

df['FamilySize'] = df['SibSp'] + df['Parch']
df = df.drop(columns=['PassengerId', 'Name', 'Cabin', 'Ticket'])

features = ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'FamilySize']
X = df[features]
y = df['Survived']


num_features = ['Age', 'Fare', 'FamilySize']
cat_features = ['Pclass', 'Sex', 'Embarked']

num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])


pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', LogisticRegression(max_iter=200))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pipeline.fit(X_train, y_train)

def predict_survival(pclass, sex, age, fare, embarked, sibsp, parch):
    family_size = sibsp + parch
    input_df = pd.DataFrame([{
        'Pclass': pclass,
        'Sex': sex,
        'Age': age,
        'Fare': fare,
        'Embarked': embarked,
        'FamilySize': family_size
    }])

    pred = pipeline.predict(input_df)[0]
    prob = pipeline.predict_proba(input_df)[0][1]

    status = "✅ Likely Survived" if pred == 1 else "❌ Likely Not Survived"

    if prob > 0.85:
        risk = "🟢 Very High Chance of Survival"
    elif prob > 0.65:
        risk = "🟡 Moderate Chance of Survival"
    elif prob > 0.5:
        risk = "🟠 Slight Chance of Survival"
    else:
        risk = "🔴 Low Chance of Survival"

    return f"{status}\n{risk}\nConfidence: {prob*100:.1f}%"


gr.Interface(
    fn=predict_survival,
    inputs=[
        gr.Dropdown([1, 2, 3], label="Ticket Class"),
        gr.Radio(["male", "female"], label="Gender"),
        gr.Slider(0, 100, step=1, label="Age"),
        gr.Slider(0, 600, step=5, label="Fare Paid"),
        gr.Radio(["S", "C", "Q"], label="Port of Embarkation"),
        gr.Slider(0, 10, step=1, label="No. of Siblings/Spouses Aboard"),
        gr.Slider(0, 10, step=1, label="No. of Parents/Children Aboard")
    ],
    outputs="text",
    title="🚢 Titanic Survival Risk Estimator",
    description="Input passenger details to estimate survival likelihood and confidence. Built with Logistic Regression.",
    theme="soft"
).launch()


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bb604f8144c518ab7d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
